In [3]:
import os
from dotenv import load_dotenv
load_dotenv()

print("Api key is loaded")

Api key is loaded


In [4]:
from langchain_community.document_loaders import PyPDFLoader   

PDF_PATH = "telecom_guide.pdf"
loader = PyPDFLoader(PDF_PATH)
pages = loader.load()

print (f"Loaded {len(pages)} pages from the PDF.")
print ("\n--- First page preview (first 500 chars) ---")
print (pages[0].page_content[:500])


C:\Users\z004f2sd\AppData\Local\Temp\ipykernel_26304\2131896144.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
c:\D-Drive\Z004f2SD\Devops\agentic-ai-tutorial\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 9 pages from the PDF.

--- First page preview (first 500 chars) ---
Telecom Technical Reference Guide  - Internal Use Only
Telecom Technical
Reference Guide
Customer Care & Network Operations Edition
Version 3.2  |  Covers 2G / 3G / 4G LTE / 5G
Page 1


In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=100,
    separators=["\n\n", "\n", " ", ""]
)

chunks = splitter.split_documents(pages)
print (f"Split into {len(chunks)} chunks from {len(pages)} pages.")
print(f"Avg chunk length: {sum(len(c.page_content) for c in chunks) // len(chunks)} chars")
print("\n--- Example chunk ---")
print(chunks[5].page_content)


Split into 37 chunks from 9 pages.
Avg chunk length: 504 chars

--- Example chunk ---
Telecom Technical Reference Guide  - Internal Use Only
2. Troubleshooting Connectivity Issues
Connectivity problems are the most common category of customer complaints. A structured diagnostic approach
resolves the majority of cases without escalation.
Step 1  - Verify signal strength. Open the device's status bar or dial *3001#12345#* (iOS) or use a network signal
app (Android) to view the raw signal level in dBm. A signal above -85 dBm is good; between -85 and -100 dBm is
marginal; below -100 dBm is poor. If signal is weak, moving closer to a window or to a higher floor often helps.


In [7]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# Step 1: Load the embedding model. No chunks are converted on this line.
print("Loading embedding model (downloads ~90 MB on first run)...")
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Step 2: This line performs two operations internally:
#   a. The embedding model converts each chunk's page_content into a numerical vector.
#   b. ChromaDB stores each vector together with its original text and metadata.
print("Converting chunks to embeddings and storing them in ChromaDB...")
vector_db = Chroma.from_documents(chunks, embeddings)

print(f"Vector store ready. {vector_db._collection.count()} vectors stored.")

Loading embedding model (downloads ~90 MB on first run)...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7162.02it/s]


Converting chunks to embeddings and storing them in ChromaDB...
Vector store ready. 37 vectors stored.


In [8]:
retriever = vector_db.as_retriever(search_kwargs={"k": 3})

test_query = "What is VoLTE and how does it improve call quality?"
retrieved_docs = retriever.invoke(test_query)

print(f"Query: {test_query}")
print(f"Retrieved {len(retrieved_docs)} chunks:\n")
for i, doc in enumerate(retrieved_docs, 1):
    print(f"--- Chunk {i} ---")
    print(doc.page_content[:300])
    print()

Query: What is VoLTE and how does it improve call quality?
Retrieved 3 chunks:

--- Chunk 1 ---
Telecom Technical Reference Guide  - Internal Use Only
6. VoLTE, VoWiFi, and Advanced Voice Services
Voice over LTE (VoLTE) and Voice over Wi-Fi (VoWiFi) are IP-based voice technologies that replace the legacy
circuit-switched voice channel used in 2G and 3G networks.
VoLTE: With VoLTE, voice calls 

--- Chunk 2 ---
voice simultaneously without degradation. VoLTE requires a compatible device, a VoLTE-enabled SIM, and an
account that has VoLTE activated.
Enabling VoLTE: On most Android devices navigate to Settings > Mobile Network > VoLTE and toggle it on. On
iPhone go to Settings > Mobile Data > Mobile Data Opt

--- Chunk 3 ---
prioritised over general data traffic. This prevents voice quality degradation during periods of network congestion.
Without QoS, voice packets would compete with video streaming and file downloads, causing jitter and packet
loss.
Fallback Behaviour: If a VoLTE call c

In [9]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_groq import ChatGroq


# --- Helper: join retrieved chunks into a single context string ---
def format_docs(docs):
    return "\n\n---\n\n".join(doc.page_content for doc in docs)


# --- System prompt: ground the LLM in the retrieved context ---
SYSTEM_PROMPT = """\
You are a helpful telecom assistant.
Answer the question using ONLY the context provided below.
If the context does not contain enough information, say so clearly.

Context:
{context}
"""
prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "{question}"),
])

# --- Low-cost LLM via Groq API ---
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0,
)

# --- Assemble the chain ---
chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG chain assembled with a low-cost model.")

RAG chain assembled with a low-cost model.


In [10]:
question = "How does international roaming work and what charges should I expect?"

print(f"Q: {question}\n")
print("A:", chain.invoke(question))

Q: How does international roaming work and what charges should I expect?

A: **How international roaming works**

1. **Connection to a partner network** – When you leave your home network’s coverage area, your phone automatically looks for a partner (visited) network in the country you’re in.  
2. **Authentication & authorization** – The visited network uses an inter‑operator signalling protocol (SS7 or Diameter) to verify your identity. Your home network then validates your subscription and authorises the services you’re allowed to use.  
3. **Tunnelling for billing** – All of your data, voice, and SMS traffic is routed back through a secure tunnel to your home network so that it can be billed correctly. This extra hop adds a bit of latency compared with local network usage.

**What charges to expect**

| Roaming Zone | Typical cost structure | Recommendation |
|--------------|------------------------|----------------|
| **Zone A** (EU, UK, Australia, New Zealand) | Lowest roaming rat

In [11]:
print("Telecom RAG Assistant — type 'quit' to exit\n")

while True:
    question = input("Your question: ").strip()
    if question.lower() in ("quit", "exit", "q"):
        print("Goodbye!")
        break
    if not question:
        continue

    print("\nAnswer:")
    for chunk in chain.stream(question):
        print(chunk, end="", flush=True)
    print("\n")

Telecom RAG Assistant — type 'quit' to exit


Answer:
During a billing dispute the agent follows the steps outlined in the billing‑system architecture:

1. **Identify the disputed charge** – The customer reports a specific line item (e.g., a duplicate charge or an unexpected fee).  
2. **Retrieve the relevant CDRs** – The billing engine’s mediation layer has already normalised and enriched the event records. The agent pulls the Call Detail Records that correspond to the disputed event(s).  
3. **Check for duplicate processing** – For duplicate‑charge disputes the system looks for idempotency keys that should have prevented a second posting. If the key is missing or the engine failed to detect the duplicate, the charge is flagged.  
4. **Validate the rate plan** – The billing engine applies the customer’s rate plan to each event. The agent confirms that the correct rate was used.  
5. **Resolve the dispute** –  
   * If a duplicate is confirmed, the agent manually reverses the extra cha

In [12]:
debug_question = "What security measures protect against SIM swap fraud?"

docs = retriever.invoke(debug_question)
print(f"Question: {debug_question}")
print(f"Retrieved {len(docs)} chunks:\n")
for i, doc in enumerate(docs, 1):
    print(f"{'='*60}")
    print(f"Chunk {i} (page {doc.metadata.get('page', '?')})")
    print(f"{'='*60}")
    print(doc.page_content)
    print()

print("\nFinal Answer:")
print(chain.invoke(debug_question))

Question: What security measures protect against SIM swap fraud?
Retrieved 3 chunks:

Chunk 1 (page 8)
mitigate this with SS7 firewalls and anomaly detection systems, but the risk cannot be fully eliminated on legacy
protocols. 5G's use of HTTPS-based APIs (Service Based Architecture) substantially reduces this attack surface.
SIM Swap Fraud: Described in Section 5. Key mitigation: enforce strict in-person or multi-factor remote identity
verification before any SIM replacement. Flag accounts with recent SIM swaps for elevated fraud monitoring for
30 days.
International Revenue Share Fraud (IRSF): Fraudsters compromise a PBX or customer account and generate

Chunk 2 (page 5)
provide an additional layer of protection; after three incorrect PIN attempts the SIM is locked and requires a PUK
code to unlock.
SIM Swap Fraud: SIM swap attacks occur when a fraudster convinces a carrier to transfer a victim's number to a
new SIM. This allows the attacker to intercept SMS-based two-factor authent